# Reasoning vs Traditional RAG: The Evolution of Search

This notebook demonstrates the fundamental limitations of traditional keyword-based and semantic search, and shows how **Reasoning Retrievers** represent the next evolution in information retrieval.

## Key Evolution Timeline
1. **Traditional RAG** → Keyword matching, basic semantic similarity
2. **ColBERT** → Token-level late interaction, MaxSim operation  
3. **Reasoning Retrievers** → Instruction following + reasoning chains + explainable results

## What We'll Demonstrate
- Complex multi-constraint queries that traditional methods can't handle
- Contextual inference and temporal reasoning capabilities
- Negation handling and conditional logic
- The power of instruction-following retrieval

In [ ]:
# Setup and imports
import sys
sys.path.append('/Users/luvsuneja/Documents/repos/advanced-rag-experimentation/')
from setup import *

import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import re
from typing import List, Dict, Tuple

## Load Reasoning-Optimized Dataset

We're using a specially crafted dataset of restaurant reviews designed to showcase reasoning capabilities. Each review tests specific reasoning requirements that traditional search fails on.

In [ ]:
# Load the reasoning-optimized restaurant reviews
reasoning_reviews_path = os.path.join(os.getenv('DATA_DIR'), 'reasoning_restaurant_reviews.csv')
test_queries_path = os.path.join(os.getenv('DATA_DIR'), 'reasoning_test_queries.json')

reviews_df = pd.read_csv(reasoning_reviews_path)
with open(test_queries_path, 'r') as f:
    test_queries = json.load(f)

print(f"📊 Loaded {len(reviews_df)} reasoning-optimized reviews")
print(f"📝 Loaded {sum(len(queries) for queries in test_queries.values())} test queries")
print(f"\n🧪 Query categories: {list(test_queries.keys())}")

In [ ]:
# Display sample reviews by reasoning category
pd.set_option('display.max_colwidth', 100)
print("Sample reviews by reasoning category:\n")

for category in reviews_df['query_category'].unique()[:3]:
    sample = reviews_df[reviews_df['query_category'] == category].iloc[0]
    print(f"🏷️  {category.upper()}:")
    print(f"   Restaurant: {sample['restaurant']}")
    print(f"   Review: {sample['review'][:200]}...")
    print()

## Traditional Search Implementation

Let's implement traditional keyword-based and semantic search methods to establish our baseline.

In [ ]:
class TraditionalSearch:
    def __init__(self, reviews_df):
        self.reviews_df = reviews_df
        # Load semantic model for embedding-based search
        self.semantic_model = SentenceTransformer('all-MiniLM-L6-v2')
        
        # Pre-compute embeddings for all reviews
        print("Computing review embeddings...")
        self.review_embeddings = self.semantic_model.encode(
            reviews_df['review'].tolist(), 
            show_progress_bar=True
        )
    
    def keyword_search(self, query: str, top_k: int = 3) -> List[Dict]:
        """Simple keyword matching search"""
        query_words = set(query.lower().split())
        
        scores = []
        for idx, review in enumerate(self.reviews_df['review']):
            review_words = set(review.lower().split())
            # Simple overlap score
            overlap = len(query_words.intersection(review_words))
            scores.append((idx, overlap))
        
        # Sort by score and return top k
        scores.sort(key=lambda x: x[1], reverse=True)
        
        results = []
        for idx, score in scores[:top_k]:
            results.append({
                'restaurant': self.reviews_df.iloc[idx]['restaurant'],
                'review': self.reviews_df.iloc[idx]['review'],
                'score': score,
                'method': 'keyword'
            })
        
        return results
    
    def semantic_search(self, query: str, top_k: int = 3) -> List[Dict]:
        """Embedding-based semantic search"""
        # Encode query
        query_embedding = self.semantic_model.encode([query])
        
        # Compute similarities
        similarities = cosine_similarity(query_embedding, self.review_embeddings)[0]
        
        # Get top k results
        top_indices = similarities.argsort()[-top_k:][::-1]
        
        results = []
        for idx in top_indices:
            results.append({
                'restaurant': self.reviews_df.iloc[idx]['restaurant'],
                'review': self.reviews_df.iloc[idx]['review'],
                'score': similarities[idx],
                'method': 'semantic'
            })
        
        return results

# Initialize traditional search
traditional_search = TraditionalSearch(reviews_df)

## Reasoning Search Simulation

For this demo, we'll simulate reasoning search using our ground truth data to show what reasoning retrievers can achieve. In the next notebooks, we'll implement actual Promptriever and Rank1 models.

In [ ]:
class ReasoningSearchSimulator:
    """Simulates reasoning search using ground truth to show potential"""
    
    def __init__(self, reviews_df, test_queries):
        self.reviews_df = reviews_df
        self.test_queries = test_queries
        
        # Create lookup for expected matches
        self.query_to_expected = {}
        for category, queries in test_queries.items():
            for q in queries:
                self.query_to_expected[q['query']] = q['expected_matches']
    
    def reasoning_search(self, query: str, top_k: int = 3) -> List[Dict]:
        """Simulate perfect reasoning search using ground truth"""
        expected_matches = self.query_to_expected.get(query, [])
        
        if not expected_matches:
            return []
        
        results = []
        for restaurant_name in expected_matches:
            matching_review = self.reviews_df[self.reviews_df['restaurant'] == restaurant_name]
            if not matching_review.empty:
                results.append({
                    'restaurant': restaurant_name,
                    'review': matching_review.iloc[0]['review'],
                    'score': 1.0,  # Perfect reasoning match
                    'method': 'reasoning',
                    'reasoning': f"✅ Reasoning: Verified all query constraints are met"
                })
        
        return results[:top_k]

reasoning_search = ReasoningSearchSimulator(reviews_df, test_queries)

## Comparison Demo: Multi-Constraint Query

Let's test a complex business lunch query that requires verifying multiple constraints simultaneously.

In [ ]:
# Complex multi-constraint query
business_query = "Find restaurants suitable for a business lunch where I can discuss confidential information, accommodate my client's vegetarian diet, and stay within a $40 per person budget"

print(f"🔍 QUERY: {business_query}\n")

# Test all three methods
keyword_results = traditional_search.keyword_search(business_query, top_k=2)
semantic_results = traditional_search.semantic_search(business_query, top_k=2)
reasoning_results = reasoning_search.reasoning_search(business_query, top_k=2)

def display_results(results, method_name):
    print(f"📊 {method_name.upper()} RESULTS:")
    if not results:
        print("   ❌ No results found\n")
        return
    
    for i, result in enumerate(results, 1):
        print(f"   {i}. {result['restaurant']} (score: {result['score']:.3f})")
        if 'reasoning' in result:
            print(f"      {result['reasoning']}")
        print(f"      Review: {result['review'][:200]}...\n")

display_results(keyword_results, "Keyword Search")
display_results(semantic_results, "Semantic Search") 
display_results(reasoning_results, "Reasoning Search")

## Analysis: Why Traditional Methods Fail

Let's analyze what happened with that business lunch query.

In [ ]:
# Get the query details from our test set
business_query_details = None
for category, queries in test_queries.items():
    for q in queries:
        if "business lunch" in q['query'] and "confidential" in q['query']:
            business_query_details = q
            break

if business_query_details:
    print("🧠 REASONING ANALYSIS:\n")
    print(f"✅ Expected Match: {business_query_details['expected_matches']}")
    print(f"❌ Traditional Fails: {business_query_details['traditional_fails_because']}")
    print(f"🎯 Reasoning Succeeds: {business_query_details['reasoning_succeeds_because']}\n")
    
    # Show the correct review
    correct_restaurant = business_query_details['expected_matches'][0]
    correct_review = reviews_df[reviews_df['restaurant'] == correct_restaurant]['review'].iloc[0]
    
    print(f"📝 CORRECT REVIEW ({correct_restaurant}):")
    print(f"{correct_review}")
    
    print("\n🔍 CONSTRAINT VERIFICATION:")
    print("✅ Confidential discussions: 'private booths with high backs ensure conversations stay confidential'")
    print("✅ Vegetarian options: 'vegetarian menu is surprisingly extensive with gourmet options'")
    print("✅ Budget constraint: 'keeps everything under $35 per person including appetizer'")

## Demo 2: Negation and Exclusion Query

Traditional search is notoriously bad at negation. Let's test a query about restaurants that explicitly exclude large groups.

In [ ]:
# Negation query
negation_query = "Find restaurants that explicitly mention they are NOT suitable for large groups"

print(f"🔍 NEGATION QUERY: {negation_query}\n")

# Test all methods
keyword_neg = traditional_search.keyword_search(negation_query, top_k=2)
semantic_neg = traditional_search.semantic_search(negation_query, top_k=2)
reasoning_neg = reasoning_search.reasoning_search(negation_query, top_k=2)

display_results(keyword_neg, "Keyword Search")
display_results(semantic_neg, "Semantic Search")
display_results(reasoning_neg, "Reasoning Search")

In [ ]:
# Analyze the negation results
negation_details = None
for queries in test_queries['negation_exclusion_queries']:
    if "NOT suitable for large groups" in queries['query']:
        negation_details = queries
        break

if negation_details:
    print("🧠 NEGATION ANALYSIS:\n")
    print(f"✅ Expected: {negation_details['expected_matches']}")
    print(f"❌ Traditional fails: {negation_details['traditional_fails_because']}")
    print(f"🎯 Reasoning succeeds: {negation_details['reasoning_succeeds_because']}\n")
    
    # Show the correct review
    correct_restaurant = negation_details['expected_matches'][0]
    correct_review = reviews_df[reviews_df['restaurant'] == correct_restaurant]['review'].iloc[0]
    
    print(f"📝 CORRECT REVIEW ({correct_restaurant}):")
    print(f"{correct_review}\n")
    
    print("🔍 KEY NEGATION PHRASE:")
    print("✅ 'this is NOT a place for groups larger than 4'")

## Demo 3: Temporal Reasoning Query

Let's test understanding of change over time - restaurants that have declined recently.

In [ ]:
# Temporal reasoning query
temporal_query = "Find restaurants that were good in the past but have declined recently"

print(f"🔍 TEMPORAL QUERY: {temporal_query}\n")

# Test all methods
keyword_temp = traditional_search.keyword_search(temporal_query, top_k=2)
semantic_temp = traditional_search.semantic_search(temporal_query, top_k=2)
reasoning_temp = reasoning_search.reasoning_search(temporal_query, top_k=2)

display_results(keyword_temp, "Keyword Search")
display_results(semantic_temp, "Semantic Search")
display_results(reasoning_temp, "Reasoning Search")

In [ ]:
# Analyze temporal reasoning
temporal_details = test_queries['temporal_reasoning_queries'][0]

print("🧠 TEMPORAL REASONING ANALYSIS:\n")
print(f"✅ Expected: {temporal_details['expected_matches']}")
print(f"❌ Traditional fails: {temporal_details['traditional_fails_because']}")
print(f"🎯 Reasoning succeeds: {temporal_details['reasoning_succeeds_because']}\n")

# Show examples of temporal language
for restaurant in temporal_details['expected_matches']:
    review = reviews_df[reviews_df['restaurant'] == restaurant]['review'].iloc[0]
    print(f"📝 {restaurant}:")
    
    # Extract temporal phrases
    temporal_phrases = []
    if "used to" in review.lower():
        temporal_phrases.append("'used to be our go-to'")
    if "six months ago" in review.lower():
        temporal_phrases.append("'six months ago'")
    if "past its prime" in review.lower():
        temporal_phrases.append("'past its prime'")
    if "two years ago" in review.lower():
        temporal_phrases.append("'two years ago'")
        
    print(f"   🕐 Temporal phrases: {', '.join(temporal_phrases)}")
    print(f"   📄 Review: {review[:150]}...\n")

## Performance Comparison Summary

Let's run a comprehensive evaluation across all query types to show the dramatic improvement reasoning brings.

In [ ]:
def evaluate_search_methods():
    """Comprehensive evaluation across all query types"""
    results = {
        'keyword': {'correct': 0, 'total': 0, 'categories': {}},
        'semantic': {'correct': 0, 'total': 0, 'categories': {}},
        'reasoning': {'correct': 0, 'total': 0, 'categories': {}}
    }
    
    print("🧪 COMPREHENSIVE EVALUATION\n")
    
    for category, queries in test_queries.items():
        print(f"📊 Category: {category.replace('_', ' ').title()}")
        
        category_results = {'keyword': 0, 'semantic': 0, 'reasoning': 0}
        
        for query_data in queries:
            query = query_data['query']
            expected = set(query_data['expected_matches'])
            
            # Test each method
            keyword_res = traditional_search.keyword_search(query, top_k=3)
            semantic_res = traditional_search.semantic_search(query, top_k=3)
            reasoning_res = reasoning_search.reasoning_search(query, top_k=3)
            
            # Check if correct restaurant is in top results
            keyword_found = any(r['restaurant'] in expected for r in keyword_res)
            semantic_found = any(r['restaurant'] in expected for r in semantic_res)
            reasoning_found = any(r['restaurant'] in expected for r in reasoning_res)
            
            # Update counters
            results['keyword']['total'] += 1
            results['semantic']['total'] += 1
            results['reasoning']['total'] += 1
            
            if keyword_found:
                results['keyword']['correct'] += 1
                category_results['keyword'] += 1
            if semantic_found:
                results['semantic']['correct'] += 1
                category_results['semantic'] += 1
            if reasoning_found:
                results['reasoning']['correct'] += 1
                category_results['reasoning'] += 1
        
        # Store category results
        total_queries = len(queries)
        results['keyword']['categories'][category] = category_results['keyword'] / total_queries
        results['semantic']['categories'][category] = category_results['semantic'] / total_queries
        results['reasoning']['categories'][category] = category_results['reasoning'] / total_queries
        
        print(f"   Keyword: {category_results['keyword']}/{total_queries} ({category_results['keyword']/total_queries*100:.0f}%)")
        print(f"   Semantic: {category_results['semantic']}/{total_queries} ({category_results['semantic']/total_queries*100:.0f}%)")
        print(f"   Reasoning: {category_results['reasoning']}/{total_queries} ({category_results['reasoning']/total_queries*100:.0f}%)\n")
    
    return results

evaluation_results = evaluate_search_methods()

In [ ]:
# Create visualization of results
plt.figure(figsize=(12, 8))

# Overall accuracy comparison
plt.subplot(2, 2, 1)
methods = ['Keyword', 'Semantic', 'Reasoning']
accuracies = [
    evaluation_results['keyword']['correct'] / evaluation_results['keyword']['total'] * 100,
    evaluation_results['semantic']['correct'] / evaluation_results['semantic']['total'] * 100,
    evaluation_results['reasoning']['correct'] / evaluation_results['reasoning']['total'] * 100
]

bars = plt.bar(methods, accuracies, color=['#ff6b6b', '#4ecdc4', '#45b7d1'])
plt.title('Overall Accuracy Comparison', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy (%)')
plt.ylim(0, 100)

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
             f'{acc:.0f}%', ha='center', va='bottom', fontweight='bold')

# Category breakdown
plt.subplot(2, 2, 2)
categories = list(evaluation_results['reasoning']['categories'].keys())
category_names = [cat.replace('_queries', '').replace('_', '\n').title() for cat in categories]

reasoning_scores = [evaluation_results['reasoning']['categories'][cat] * 100 for cat in categories]
semantic_scores = [evaluation_results['semantic']['categories'][cat] * 100 for cat in categories]

x = np.arange(len(categories))
width = 0.35

plt.bar(x - width/2, semantic_scores, width, label='Semantic', alpha=0.7, color='#4ecdc4')
plt.bar(x + width/2, reasoning_scores, width, label='Reasoning', alpha=0.7, color='#45b7d1')

plt.title('Accuracy by Query Type', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy (%)')
plt.xticks(x, category_names, rotation=45, ha='right', fontsize=8)
plt.legend()
plt.ylim(0, 100)

# Query complexity chart
plt.subplot(2, 1, 2)
complexity_order = [
    'contextual_inference_queries',
    'multi_constraint_queries', 
    'conditional_logic_queries',
    'temporal_reasoning_queries',
    'negation_exclusion_queries',
    'comparative_reasoning_queries',
    'meta_reasoning_queries'
]

complexity_names = [name.replace('_queries', '').replace('_', ' ').title() for name in complexity_order]
complexity_reasoning = [evaluation_results['reasoning']['categories'][cat] * 100 for cat in complexity_order if cat in evaluation_results['reasoning']['categories']]
complexity_semantic = [evaluation_results['semantic']['categories'][cat] * 100 for cat in complexity_order if cat in evaluation_results['semantic']['categories']]

x_complex = np.arange(len(complexity_names))
plt.plot(x_complex, complexity_semantic[:len(x_complex)], 'o-', label='Semantic Search', linewidth=2, markersize=8, color='#4ecdc4')
plt.plot(x_complex, complexity_reasoning[:len(x_complex)], 'o-', label='Reasoning Search', linewidth=2, markersize=8, color='#45b7d1')

plt.title('Performance vs Query Complexity', fontsize=14, fontweight='bold')
plt.xlabel('Query Complexity (Left to Right: Simple → Complex)')
plt.ylabel('Accuracy (%)')
plt.xticks(x_complex, complexity_names, rotation=45, ha='right')
plt.legend()
plt.ylim(0, 110)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
print("\n📊 FINAL PERFORMANCE SUMMARY")
print("=" * 50)
for method in ['keyword', 'semantic', 'reasoning']:
    accuracy = evaluation_results[method]['correct'] / evaluation_results[method]['total'] * 100
    print(f"{method.title()} Search: {accuracy:.1f}% accuracy ({evaluation_results[method]['correct']}/{evaluation_results[method]['total']} correct)")

improvement = accuracies[2] - accuracies[1]  # reasoning - semantic
print(f"\n🚀 Reasoning delivers {improvement:.1f} percentage point improvement over semantic search!")

## Key Takeaways

This comparison reveals the fundamental limitations of traditional search methods:

### ❌ Traditional Search Fails Because:
1. **Multi-constraint queries**: Can't verify ALL constraints simultaneously
2. **Negation handling**: Notoriously bad at "NOT" conditions
3. **Temporal reasoning**: Can't understand change over time
4. **Conditional logic**: Struggles with "if-then" relationships
5. **Context inference**: Misses nuanced meanings

### ✅ Reasoning Search Succeeds By:
1. **Instruction following**: Understands complex natural language requirements
2. **Constraint verification**: Can check multiple conditions systematically
3. **Explainable results**: Provides reasoning chains for decisions
4. **Context awareness**: Grasps subtle implications and relationships
5. **Temporal understanding**: Tracks changes and improvements over time

## What's Next?

In the next notebooks, we'll implement:
- **Notebook 2**: Promptriever for instruction-following retrieval
- **Notebook 3**: Rank1 for test-time reasoning with explainable chains

These represent the cutting edge of retrieval technology in 2025!